# 必要パッケージのインポート

In [ ]:
from elasticsearch8 import Elasticsearch
from sentence_transformers import SentenceTransformer
import time

# 設定

In [ ]:
ES_URL = 'http://llm-rag-examples-elasticsearch1:9200'
INDEX_NAME = 'vector_test02'

In [ ]:
MODEL_NAME = 'all-MiniLM-L6-v2'
MODEL_DIM = 384

# 初期化

In [ ]:
# 初期化
es = Elasticsearch(ES_URL)
model = SentenceTransformer(MODEL_NAME)

# インデックス作成

In [ ]:
if es.indices.exists(index=INDEX_NAME):
    print(f"Index '{INDEX_NAME}' is exists.")
else:
    mapping = {
        'mappings': {
            'properties': {
                'text': {'type': 'text'},
                'vector': {
                    'type': 'dense_vector',
                    'dims': MODEL_DIM,    # モデルの次元数
                    'index': True,
                    'similarity': 'cosine'
                }
            }
        }
    }
    es.indices.create(index=INDEX_NAME, body=mapping)
    print(f"Index '{INDEX_NAME}' created.")

In [ ]:
import requests

In [ ]:
res = requests.head(f"{ES_URL}/{INDEX_NAME}")
exists_index = res.status_code == 200
exists_index

In [ ]:
es = Elasticsearch(
    [ES_URL],
    headers={
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
)

In [ ]:
if exists_index:
    print(f"Index '{INDEX_NAME}' already exists.")
else:
    mapping = {
        "mappings": {
            "properties": {
                "text": {"type": "text"},
                "vector": {
                    "type": "dense_vector",
                    "dims": 384,
                    "index": True,
                    "similarity": "cosine"
                }
            }
        }
    }
    res = requests.put(f"{ES_URL}/{INDEX_NAME}", headers=HEADERS, data=json.dumps(mapping))
    res.raise_for_status()
    print(f"Index '{INDEX_NAME}' created.")

In [ ]:
# try:
#     es.indices.get(index=INDEX_NAME, request_timeout=30)
#     print(f"Index '{INDEX_NAME}' already exists.")
# except NotFoundError:
#     es.indices.create(
#         index=index_name,
#         mappings={
#             "properties": {
#                 "text":   {"type": "text"},
#                 "vector": {"type": "dense_vector", "dims": VECTOR_DIM}
#             }
#         },
#         request_timeout=30
#     )
#     print(f"Created index '{INDEX_NAME}'.")
# except (ApiError, TransportError) as e:
#     print(f"Error while checking/creating index: {e}")

# ドキュメントをインデックス

In [ ]:
# --- 登録するテキストデータ ---
texts = [
    '猫は可愛い動物です。',
    '犬は人間の親友と呼ばれています。',
    '東京は日本の首都です。'
]

In [ ]:
for i, text in enumerate(texts):
    vector = model.encode(text)
    doc = {
        'text': text,
        'vector': vector.tolist()
    }
    es.index(index=INDEX_NAME, id=i, document=doc)

print('データ登録完了！')

# ベクトル検索

In [ ]:
QUERY_TEXT = '日本の都市'

In [ ]:
query_vector = model.encode(QUERY_TEXT)

In [ ]:
search_body = {
    'size': 3,
    'knn': {
        'field': 'vector',
        'query_vector': query_vector.tolist(),
        'k': 3,
        'num_candidates': 100
    }
}

In [ ]:
# search_body

In [ ]:
response = es.search(index=INDEX_NAME, body=search_body)

In [ ]:
# response

In [ ]:
time.sleep(5)

In [ ]:
# --- 検索結果表示 ---
for hit in response['hits']['hits']:
    print(f'スコア: {hit['_score']:.4f} テキスト: {hit['_source']['text']}')